# BirdCLEF 2026 — Submission: best-derived-v1

**Model:** Google Perch v2 + Fine-tuned Classification Head  
**Experiment:** `best-derived-v1`  
**Official ROC-AUC:** 0.5115  
**Val ROC-AUC:** 0.9750  
**Key techniques:**
- min_rating ≥ 3.0 / soundscapes in training / cosine LR / mixup / TTA

**Dataset:** Upload `checkpoints/best-derived-v1/best_head.weights.h5`  
as Kaggle dataset `birdclef2026-best-derived-v1` and set `WEIGHTS_PATH` below.


In [ ]:
# Ensure TF 2.20+ (required for Perch v2 StableHLO compatibility)
import subprocess, sys

def _tf_version():
    try:
        import tensorflow as tf
        return tuple(int(x) for x in tf.__version__.split(".")[:2])
    except Exception:
        return (0, 0)

if _tf_version() < (2, 20):
    import os
    tf_wheel = "/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"
    tb_wheel = "/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl"
    if os.path.isfile(tb_wheel):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", tb_wheel], check=False)
    if os.path.isfile(tf_wheel):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", tf_wheel], check=False)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow==2.20.0"], check=False)
    if "tensorflow" in sys.modules:
        del sys.modules["tensorflow"]

import tensorflow as tf
print(f"TF version: {tf.__version__}")
assert tuple(int(x) for x in tf.__version__.split(".")[:2]) >= (2, 20), \n    f"TF 2.20+ required for Perch StableHLO, got {tf.__version__}"


In [ ]:
import glob
import os
import re
import time

import librosa
import numpy as np
import pandas as pd
import tensorflow as tf

print('TF version:', tf.__version__)

START_TIME = time.time()
TERMINATE_TIME = START_TIME + 5_400  # 90 min safety cutoff

## Paths — adjust for your Kaggle dataset names

In [ ]:
# ── Data paths ────────────────────────────────────────────────────────────────
DATA_DIR        = '/kaggle/input/birdclef-2026'
PERCH_DIR       = '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'

# Upload checkpoints/best-derived-v1/best_head.weights.h5 as a Kaggle dataset
# Dataset name: "birdclef2026-best-derived-v1"
WEIGHTS_PATH    = '/kaggle/input/birdclef2026-best-derived-v1/best_head.weights.h5'

# Model hyperparams (must match training)
NUM_CLASSES     = 234
HIDDEN_DIM      = 512
DROPOUT         = 0.3
SAMPLE_RATE     = 32_000
CLIP_DURATION   = 5        # seconds
CLIP_SAMPLES    = SAMPLE_RATE * CLIP_DURATION

BATCH_SIZE      = 64       # increase if GPU memory allows
ENABLE_TTA      = True     # 2.5-second half-window shift TTA

# Soundscapes
TEST_DIR   = os.path.join(DATA_DIR, 'test_soundscapes')
TRAIN_SC   = os.path.join(DATA_DIR, 'train_soundscapes')
SC_DIR     = TEST_DIR if glob.glob(os.path.join(TEST_DIR, '*.ogg')) else TRAIN_SC
print(f'Soundscapes dir: {SC_DIR}')
print(f'Files: {len(glob.glob(os.path.join(SC_DIR, "*.ogg")))}')

## Model Definition (inline — no external src/ dependency)

In [ ]:
class ClassificationHead(tf.keras.Model):
    """Two-layer MLP: Perch embedding → class logits."""

    def __init__(self, num_classes: int, hidden_dim: int = 512, dropout: float = 0.3):
        super().__init__()
        self.fc1     = tf.keras.layers.Dense(hidden_dim)
        self.act     = tf.keras.layers.Activation('relu')
        self.dropout = tf.keras.layers.Dropout(dropout)
        self.fc2     = tf.keras.layers.Dense(num_classes)

    def call(self, x, training: bool = False):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x, training=training)
        return self.fc2(x)


# ── Load Perch backbone ───────────────────────────────────────────────────────
print('Loading Perch …')
perch = tf.saved_model.load(PERCH_DIR)
sig   = perch.signatures['serving_default']

# Find embedding output key
_dummy   = tf.zeros((1, CLIP_SAMPLES), tf.float32)
_outputs = sig(inputs=_dummy)
EMB_KEY  = next((k for k in ('embedding', 'embeddings', 'label', 'logits')
                 if k in _outputs), next(iter(_outputs.keys())))
EMB_DIM  = int(_outputs[EMB_KEY].shape[-1])
print(f'Perch output key={EMB_KEY!r}  dim={EMB_DIM}')


# ── Build head & load weights ────────────────────────────────────────────────
head = ClassificationHead(NUM_CLASSES, HIDDEN_DIM, DROPOUT)
# Warm-up build
_ = head(tf.zeros((1, EMB_DIM)), training=False)

# Load weights directly via h5py (avoids Keras legacy-format mismatch)
import h5py
with h5py.File(WEIGHTS_PATH, 'r') as wf:
    head.fc1.kernel.assign(wf['fc1']['vars']['0'][:])
    head.fc1.bias.assign(  wf['fc1']['vars']['1'][:])
    head.fc2.kernel.assign(wf['fc2']['vars']['0'][:])
    head.fc2.bias.assign(  wf['fc2']['vars']['1'][:])
print(f'Weights loaded ← {WEIGHTS_PATH}')
print(f'Head params: {sum(int(np.prod(v.shape)) for v in head.trainable_variables):,}')

## Species Mapping

In [ ]:
sample_sub     = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
target_species = sample_sub.columns[1:].tolist()   # 234 species codes
print(f'Target species: {len(target_species)}')
print(target_species[:5])

## Inference

In [ ]:
@tf.function(input_signature=[tf.TensorSpec([None, CLIP_SAMPLES], tf.float32)])
def predict_batch(clips):
    """Perch embedding → head logits → sigmoid probabilities."""
    emb    = tf.stop_gradient(sig(inputs=clips)[EMB_KEY])
    logits = head(emb, training=False)
    return tf.sigmoid(logits)


def infer_clips(clips: np.ndarray) -> np.ndarray:
    """Run batched inference; returns (n_clips, num_classes) float32."""
    all_preds = []
    for i in range(0, len(clips), BATCH_SIZE):
        batch = tf.constant(clips[i:i + BATCH_SIZE], dtype=tf.float32)
        all_preds.append(predict_batch(batch).numpy())
    return np.concatenate(all_preds, axis=0)


def process_soundscape(filepath: str):
    """
    Split a soundscape into 5-second clips, run inference.
    Optionally applies TTA (half-window 2.5-second shift, BirdCLEF 2025 2nd place).

    Returns
    -------
    row_ids    : list of str  e.g. ["soundscape_id_5", "_10", ...]
    preds      : np.ndarray   shape (n_segments, num_classes)
    """
    ss_id = re.sub(r'\.ogg$', '', os.path.basename(filepath), flags=re.IGNORECASE)

    audio, _ = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    audio = audio.astype(np.float32)

    n_segs = len(audio) // CLIP_SAMPLES
    if n_segs == 0:
        return [], np.empty((0, NUM_CLASSES))

    clips    = np.stack([audio[i * CLIP_SAMPLES:(i + 1) * CLIP_SAMPLES]
                         for i in range(n_segs)])
    row_ids  = [f'{ss_id}_{(i + 1) * CLIP_DURATION}' for i in range(n_segs)]
    preds    = infer_clips(clips)

    if ENABLE_TTA:
        # 2.5-second shifted clips (temporal TTA)
        half         = CLIP_SAMPLES // 2
        audio_shift  = audio[half:]
        n_shift      = len(audio_shift) // CLIP_SAMPLES
        if n_shift > 0:
            clips_shift  = np.stack([audio_shift[i * CLIP_SAMPLES:(i + 1) * CLIP_SAMPLES]
                                     for i in range(n_shift)])
            preds_shift  = infer_clips(clips_shift)
            n_use        = min(len(preds), len(preds_shift))
            preds[:n_use] = 0.5 * preds[:n_use] + 0.5 * preds_shift[:n_use]

    return row_ids, preds


# ── Main inference loop ───────────────────────────────────────────────────────
ogg_files  = sorted(glob.glob(os.path.join(SC_DIR, '*.ogg')))
all_row_ids, all_preds = [], []

for k, fpath in enumerate(ogg_files):
    if time.time() > TERMINATE_TIME:
        print(f'Time limit reached at file {k}/{len(ogg_files)} — stopping early.')
        break

    fname = os.path.basename(fpath)
    print(f'[{k+1}/{len(ogg_files)}] {fname}')

    try:
        row_ids, preds = process_soundscape(fpath)
    except Exception as e:
        print(f'  ERROR: {e}')
        continue

    if len(row_ids) > 0:
        all_row_ids.extend(row_ids)
        all_preds.append(preds)

print(f'\nInference done. {len(all_row_ids)} rows.')

## Build Submission

In [ ]:
if not all_preds:
    raise RuntimeError('No predictions generated. Check soundscapes path and model loading.')

predictions = np.concatenate(all_preds, axis=0)  # (n_rows, 234)

submission = pd.DataFrame(predictions, columns=target_species)
submission.insert(0, 'row_id', all_row_ids)

# Ensure all rows from sample_submission are present (fill missing with 0)
submission = sample_sub[['row_id']].merge(submission, on='row_id', how='left').fillna(0.0)

submission.to_csv('submission.csv', index=False)
print(f'submission.csv saved — {submission.shape[0]} rows × {len(target_species)} species')
print(f'Elapsed: {(time.time()-START_TIME)/60:.1f} min')
submission.head(10)

## Quick Validation (train soundscapes only)

Run this cell only when testing locally (not during submission).

In [ ]:
VALIDATE = False   # set True to compute ROC-AUC on train soundscapes

if VALIDATE:
    from sklearn.metrics import roc_auc_score

    labels_csv = os.path.join(DATA_DIR, 'train_soundscapes_labels.csv')
    labels_df  = pd.read_csv(labels_csv)

    # Build ground-truth matrix
    def _end_to_row_id(row):
        fname = re.sub(r'\.ogg$', '', row['filename'], flags=re.IGNORECASE)
        h, m, s = str(row['end']).split(':')
        return f"{fname}_{int(h)*3600+int(m)*60+int(s)}"

    labels_df['row_id'] = labels_df.apply(_end_to_row_id, axis=1)
    gt = pd.pivot_table(
        labels_df, index='row_id', columns='primary_label',
        aggfunc='size', fill_value=0
    ).clip(upper=1).reindex(columns=target_species, fill_value=0)

    pred_df = submission.set_index('row_id')[target_species]
    common  = gt.index.intersection(pred_df.index)

    y_true = gt.loc[common].values
    y_pred = pred_df.loc[common].values

    scored_mask = y_true.sum(axis=0) > 0
    roc = roc_auc_score(y_true[:, scored_mask], y_pred[:, scored_mask], average='macro')
    print(f'\nLocal ROC-AUC (macro, {scored_mask.sum()} scored species): {roc:.4f}')